# Benchmark summary tables

Scores all methods and draws one table per cohort plus a combined table.

- Metrics are averaged over samples, then min–max scaled per cohort across non-control methods.
- MECR is size-adjusted per cohort and inverted. Cell density is log-scaled.
- The combined table averages scaled metrics across cohorts.
- The total score is the mean of the available categories.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[0]))
from cellseg_benchmark._constants import BASE_PATH, celltype_groups, main_methods
from cellseg_benchmark.metrics.benchmark_table import (
    build_benchmark_tables,
    metrics,
    plot_benchmark_table,
)

cohorts = ["aging", "ABCAtlas", "VizgenMouseBrain"]
out_dir = Path(BASE_PATH) / "metrics" / "merged" / "benchmark_tables"
out_dir.mkdir(parents=True, exist_ok=True)

## Build tables

In [ ]:
tables, pooled_score = build_benchmark_tables(cohorts)

## Plot tables

In [ ]:
for name, table in tables.items():
    plot_benchmark_table(table[table.index.isin(main_methods)], out_dir / f"{name}_main")
    plot_benchmark_table(table, out_dir / f"{name}_full")

df_expr = tables["combined"].assign(total_score=lambda df: df.expr_score).sort_values("total_score", ascending=False)
plot_benchmark_table(df_expr, out_dir / "combined_expression", score_label="Expr. score")

## Cell-type tables

Same scores restricted to the cell types of each group in `celltype_groups`, scaled with the min, max and MECR fit of the all-cells table of the same cohort. Silhouette only for groups with more than one cell type.

In [ ]:
for group in celltype_groups:
    group_tables, _ = build_benchmark_tables(cohorts, group)
    for name, table in group_tables.items():
        plot_benchmark_table(table[table.index.isin(main_methods)], out_dir / f"{name}_{group}_main")
        plot_benchmark_table(table, out_dir / f"{name}_{group}_full")

## Sensitivity checks

In [9]:
real = tables["combined"].query("~is_control")
cohort_scores = pd.DataFrame({c: tables[c].total_score for c in cohorts}).reindex(real.index)
mean_rank = cohort_scores.rank(ascending=False).mean(axis=1)

print("shared range and MECR fit:", real.total_score.corr(pooled_score[real.index], method="spearman"))
print("mean rank across cohorts:", real.total_score.corr(-mean_rank, method="spearman"))

shared range and MECR fit: 0.9755808440018965
mean rank across cohorts: 0.9932983550273304


## Missing values

In [10]:
for name, table in tables.items():
    missing = table[metrics].isna()
    missing["ficture_f1_score"] &= ~table.na_ficture
    missing["Ovrlpy_stats_mean_integrity"] &= ~table.na_z
    print(name, {metric: missing.index[missing[metric]].tolist() for metric in metrics if missing[metric].any()})

aging {'marker_f1': ['Baysor Sopa (CP2 T, 0.8)', 'Baysor Sopa (CP2 P, 0.8)', 'Ficture Seg (CP1 3D N)', 'Raster 5μm'], 'ficture_f1_score': ['CellposeSAM Sopa', 'Cellpose1 P+N VPT', 'Baysor Sopa (CP1 P, 0.8)', 'Baysor Sopa (CP1 N, 1.0)', 'Baysor'], 'elongation': ['Baysor', 'Baysor 3D']}
ABCAtlas {'ficture_f1_score': ['CellposeSAM Sopa', 'Baysor'], 'elongation': ['Baysor 3D', 'Baysor']}
VizgenMouseBrain {'ficture_f1_score': ['CellposeSAM Sopa', 'Baysor'], 'elongation': ['Baysor', 'Baysor 3D']}
combined {'ficture_f1_score': ['CellposeSAM Sopa', 'Baysor'], 'elongation': ['Baysor', 'Baysor 3D']}
